# demo

> Render FastHTML markup in isolated previews with current styles, behaviors, canned htmx routes, and reusable component examples

In [ ]:
#| default_exp demo

`demo` renders ordinary FastHTML markup in an isolated document with the registry's current styles, behaviors, and browser helpers from `reg.hdrs()`, matching a served app. Previews work in notebooks and generated docs without an asset server.


In [ ]:
from fastcore.utils import *
from fastcore.meta import delegates
from fasthtml.common import to_xml, Div, Iframe, Button, Html, Head, Body, Style, Script, FT
from fastcore.xml import highlight
from fasthtml.core import def_hdrs
from answerui.registry import reg, Hyperelement
import answerui.hdrs  # installs Registry.hdrs
from fastlucide.icons import spritesheet

In [ ]:
from answerui.registry import hypelt
from fastcore.test import *
from fasthtml.common import P, Details, Summary
from answerui.controls import Btn
from answerui.overlays import Dropdown

## Rendering previews

Each preview document includes a stylesheet, behavior script, browser helpers, the icon `spritesheet`, and htmx. FastHTML's `def_hdrs` returns the default app headers: charset and viewport meta tags, htmx 2, and FastHTML's helper script. `def_hdrs(htmx4=True)` returns htmx 4 instead with an `htmx-config` meta tag setting `metaCharacter` to `-`, matching `fast_app(htmx4=True)`. `demo_html` passes `surreal=False` to omit Surreal.

htmx comes before the behavior script in `<head>` because the behavior script registers with htmx. Both load before the page body, which includes `spritesheet`. Pass `css` to substitute a custom stylesheet without changing the registry, or `registry` to use another registry's stylesheet, behavior script, and browser helpers instead of `reg`.

`demo_html` sets inline `min-block-size: auto` on `body` to override any full-viewport minimum height rule from the stylesheet. Without that override, an auto-resizing iframe would grow without end.

When FastHTML serves a page, it renames `get`, `post`, `put`, `delete`, `patch`, and `link` to `hx-get`, `hx-post`, `hx-put`, `hx-delete`, `hx-patch`, and `href`, turning each value into a URL. In a preview, `demo_html` renames those same attributes while keeping each path unchanged. Pass `routes` as a dict mapping `'METHOD path'` keys such as `'GET /saved'` to canned FastHTML responses. On a matching `htmx:beforeRequest` event, an inline script cancels the network request and swaps the canned markup into the target. It uses the element's inherited `hx-swap` style, defaulting to `innerHTML`.


In [ ]:
_verbs = dict(get='hx-get', post='hx-post', put='hx-put', delete='hx-delete', patch='hx-patch', link='href')
def _as_served(o):
    "`o` with `get`, `post`, `put`, `delete`, `patch` and `link` attributes renamed to `hx-get`, `hx-post`, `hx-put`, `hx-delete`, `hx-patch` and `href`, as when FastHTML serves a page"
    if isinstance(o, (list, tuple)): return tuple(_as_served(c) for c in o)
    if not isinstance(o, FT): return o
    for k,v in _verbs.items():
        if k in o.attrs: o.attrs[v] = o.attrs.pop(k)
    for c in o.children: _as_served(c)
    return o

_routes_js = r'''const routes = %s;
document.addEventListener('htmx:beforeRequest', e => {
  const r = e.detail.requestConfig, html = routes[`${r.verb.toUpperCase()} ${r.path}`];
  if (html == null) return;
  e.preventDefault();
  const spec = (htmx.closest(e.detail.elt, '[hx-swap]')?.getAttribute('hx-swap') ?? 'innerHTML').split(' ')[0];
  htmx.swap(e.detail.target, html, {swapStyle: spec});
});'''

In [ ]:
def demo_html(
    *fts,         # Markup to render
    registry=reg, # Registry providing styles, behaviors, and browser helpers
    css=None,     # Custom CSS string; None uses the styles from registry.hdrs()
    htmx4=False,  # Use htmx 4 with FastHTML's config, as `fast_app(htmx4=True)` does, instead of htmx 2
    routes=None,  # Canned htmx responses as {'METHOD path': markup}
):
    "A standalone HTML document containing markup, CSS, behaviors, browser helpers, icon spritesheet, and FastHTML's default headers"
    styles, events, browser = registry.hdrs()
    head = Head(*def_hdrs(htmx4=htmx4, surreal=False), styles if css is None else Style(css), events, browser)
    if routes: head = head(Script(_routes_js % dumps({k: to_xml(_as_served(v)) for k,v in routes.items()})))
    return to_xml(Html(head, Body(*_as_served(fts), spritesheet, style='min-block-size: auto')))


In notebooks, `demo` renders the preview inside an iframe. That isolates the preview from the host notebook's CSS, such as Solveit's styles, in both directions. The iframe's `srcdoc` attribute holds the complete document from `demo_html`. Unless you pass `height`, an inline `ResizeObserver` script resizes the iframe to fit its content. Pass `width` to preview a narrow viewport. A replay button above the iframe reloads `srcdoc` to restart dynamic components such as a self-dismissing notice.

Because the iframe contains the full stylesheet and scripts, storing its HTML in every notebook output would bloat the file. `demo` returns a `Preview` object with two representations: the iframe as `text/html` for live display, and the markup alone as `text/markdown` for saved notebooks and language models.

In [ ]:
_sizer = Script(r"new ResizeObserver(() => frameElement.style.height = document.body.scrollHeight+24+'px').observe(document.body)")

class Preview:
    "A rendered preview: the iframe as HTML for live display, and the markup as Markdown for saved notebooks and models"
    def __init__(self, fts, frame): store_attr()
    def __repr__(self): return to_xml(self.fts)
    def _repr_mimebundle_(self, include=None, exclude=None): return {'text/html': to_xml(self.frame), 'text/markdown': highlight(self.fts)}, {'nbdev': {'strip': ['text/html']}}

def demo(
    *fts,         # Markup to render
    registry=reg, # Registry providing styles, behaviors, and browser helpers
    css=None,     # Custom CSS string; None uses the styles from registry.hdrs()
    height=None,  # Fixed iframe height (e.g. '10rem') for top-layer content ResizeObserver cannot measure; None resizes to content
    width=None,   # Fixed iframe width, e.g. '24rem', to preview a narrow viewport; None fills the cell
    htmx4=False,  # Use htmx 4 with FastHTML's config, as `fast_app(htmx4=True)` does, instead of htmx 2
    routes=None,  # Canned htmx responses as {'METHOD path': markup}
):
    "Render markup in an isolated, replayable notebook preview"
    doc = fts if height is not None else fts + (_sizer,)
    style = f'width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:{height or "auto"};'
    frame = Iframe(srcdoc=demo_html(*doc, registry=registry, css=css, htmx4=htmx4, routes=routes), title='answerui preview', style=style)
    replay = Button('↻', type='button', aria_label='replay', title='replay',
        onclick=r"const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc",
        style='border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px')
    return Preview(fts, Div(Div(replay, style='text-align:right'), frame, style=f'width:{width or "100%"}'))


A preview does not require a registered component. Here `css=''` replaces the registry's stylesheet with an empty one, showing a native element exactly as the browser draws it:

In [ ]:
demo(Details(Summary('More information'), P('This disclosure is a native details element.')), css='')

```html
<details><summary>More information</summary>  <p>This disclosure is a native details element.</p>
</details>
```

Let's recreate the notice from the earlier notebooks with one `hypelt` call that registers its styles and behaviors together. Clicking its `dismiss` button removes the notice. With `autodismiss` set, the `init` handler removes the notice after that many milliseconds (4000 by default) once htmx loads it.

In [ ]:
notice_css = r"""
display: block;
padding: .75rem 1rem;
background: #f4f4f6;
border: 1px solid #ccc;
border-radius: 8px;
"""
NoticeBox = hypelt('NoticeBox', css=notice_css, layer='components', on={
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()",
    ('&[autodismiss]', 'init'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss') || 4000)); el.remove()"},
    doc='A dismissible notice.')

Click Dismiss, then click the replay button to restore the notice:

In [ ]:
demo(NoticeBox('Changes saved. ', Button('Dismiss', type='button', dismiss=True)))

```html
<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box>
```

`_repr_mimebundle_` returns both representations with `{'nbdev': {'strip': ['text/html']}}` metadata. `nbdev-clean` uses that metadata to strip the iframe HTML from saved notebook outputs and keep the Markdown:


In [ ]:
bundle, meta = demo(NoticeBox('Changes saved.'))._repr_mimebundle_()
meta

{'nbdev': {'strip': ['text/html']}}

This notice disappears after four seconds:

In [ ]:
demo(NoticeBox('This notice disappears after four seconds.', autodismiss=4000), height='4rem')

```html
<notice-box autodismiss="4000">This notice disappears after four seconds.</notice-box>
```

Here the Save button requests `/saved`. The canned response returns a `NoticeBox` and a `Dropdown` (a trigger and a panel). The routes script swaps the complete fragment into place as a served route would. Click Save, open Next action, then click the replay button to reset the preview.

In [ ]:
demo(Button('Save', type='button', get='/saved', hx_swap='outerHTML'),
    routes={'GET /saved': (NoticeBox('Saved.'), Dropdown('Next action', Btn('Keep'), Btn('Remove')))})

```html
<button type="button" hx-swap="outerHTML" hx-get="/saved">Save</button>
```

Passing `htmx4=True` includes the same htmx 4 headers and FastHTML config as `fast_app(htmx4=True)`. The behavior script runs `init` handlers through `htmx.onLoad`, which under that config listens for htmx 4's `htmx-after-process` event on newly arrived content. The notice's timer still starts:

In [ ]:
demo(NoticeBox('This notice disappears after four seconds under htmx 4.', autodismiss=4000), htmx4=True, height='4rem')

```html
<notice-box autodismiss="4000">This notice disappears after four seconds under htmx 4.</notice-box>
```

Here we pass a candidate stylesheet via `css` that replaces the notice's grey background with pale yellow. The registered behaviors still apply.

In [ ]:
candidate_css = NoticeBox.rules['base'].compiled_css.replace('#f4f4f6', '#fff4cc')
demo(NoticeBox('Trying a warmer surface. ', Button('Dismiss', type='button', dismiss=True)), css=candidate_css)

```html
<notice-box>Trying a warmer surface. <button type="button" dismiss>Dismiss</button></notice-box>
```

To register an application override for all later previews and application pages, call `NoticeBox.rule`. It adds CSS to the `app` layer without changing the notice's base definition or behaviors:

In [ ]:
NoticeBox.rule('corners', 'border-radius: 1rem;', doc='Rounder notice corners.')
demo(NoticeBox('The current stylesheet includes the application override.'))


```html
<notice-box>The current stylesheet includes the application override.</notice-box>
```

## Reusable examples

Use the `example` decorator to attach a no-argument markup function to a hyperelement as `NoticeBox.example`. Calling `NoticeBox.demo()` renders that markup with the hyperelement's registry.

Attaching another example replaces the previous one. To show multiple variants, return them together from one function.

In [ ]:
def example(h):
    "Decorator that stores a no-argument markup function as `h.example`, for `h.demo()` to render"
    def _attach(f):
        h.example = f
        return f
    return _attach

@patch
@delegates(demo, but=['registry'])
def demo(self:Hyperelement, **kwargs):
    "Render this hyperelement's example with its registry's current styles and behaviors"
    if not hasattr(self, 'example'): raise ValueError(f'{self.__name__} has no example; attach one with @example({self.__name__})')
    return demo(self.example(), registry=self.registry, **kwargs)


In [ ]:
@example(NoticeBox)
def notice_example():
    return Div(NoticeBox('Changes saved. ', Button('Dismiss', type='button', dismiss=True)),
        NoticeBox('This notice stays visible.'), style='display:grid;gap:1rem')

The example itself is ordinary markup:

In [ ]:
to_xml(NoticeBox.example())

'<div style="display:grid;gap:1rem">\n<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>\n'

`NoticeBox.demo()` renders it with the current styles and behaviors:

In [ ]:
NoticeBox.demo()


```html
<div style="display:grid;gap:1rem">
<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>

```

`NoticeBox.example()` returns fresh FastHTML markup that can be inspected or reused in a larger composition. `NoticeBox.demo()` accepts `demo`'s options, such as `height` and `css`. Each call to `NoticeBox.demo()` builds a new document from the current example, styles, and behaviors. Clicking the replay button instead restarts interactions by reloading the existing `srcdoc` in the iframe without running Python.

Because each shipped component's notebook exports its `@example` function, importing the component's module attaches the example without rendering anything.


Registering `corners` again replaces the earlier rule in place. The next `NoticeBox.demo()` builds its document with the square corners:

In [ ]:
NoticeBox.rule('corners', 'border-radius: 0;', doc='Square notice corners.')
NoticeBox.demo()

```html
<div style="display:grid;gap:1rem">
<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>

```

Calling `hypelt` again with the same name updates the existing `NoticeBox` in place, keeping its example:

In [ ]:
NoticeBox = hypelt('NoticeBox', css=notice_css, layer='components', doc='A persistent notice.')
test_is(NoticeBox.example, notice_example)
NoticeBox.demo()

```html
<div style="display:grid;gap:1rem">
<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>

```